# 01 - Prepare demand and area data

This notebook prepares the crime demand and MSOA context used by the allocation model.
It reads the forecast file, adds LSOA/MSOA information, groups crimes into buckets, and saves clean intermediate tables.

In [ ]:
import pandas as pd
from pathlib import Path
import sqlite3

## File paths

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in ["notebooks", "allocation"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
ALLOC_DB_PATH = DATA_DIR / "allocation_model.db"
PRED_DB_PATH = DATA_DIR / "police_data.db"
WALES_DB_PATH = DATA_DIR / "wales_data.db"
CCHI_GROUP_AVERAGE_PATH = DATA_DIR / "cchi_group_average.csv"
PRED_DB_URI = PRED_DB_PATH.resolve().as_uri() + "?mode=ro"
WALES_DB_URI = WALES_DB_PATH.resolve().as_uri() + "?mode=ro"


## Load forecasts and LSOA data

In [ ]:
with sqlite3.connect(PRED_DB_URI, uri=True) as conn:
    england_lsoa_info = pd.read_sql_query("SELECT * FROM lsoa_info;", conn)
    england_lsoa_demographics = pd.read_sql_query("SELECT * FROM lsoa_demographics;", conn)

with sqlite3.connect(WALES_DB_URI, uri=True) as conn:
    wales_lsoa_info = pd.read_sql_query("SELECT * FROM lsoa_info;", conn)
    wales_lsoa_demographics = pd.read_sql_query("SELECT * FROM lsoa_demographics;", conn)

lsoa_info = pd.concat([england_lsoa_info, wales_lsoa_info], ignore_index=True, sort=False)
lsoa_demographics = pd.concat(
    [england_lsoa_demographics, wales_lsoa_demographics],
    ignore_index=True,
    sort=False,
)

with sqlite3.connect(ALLOC_DB_PATH) as conn:
    forecast = pd.read_sql_query("SELECT * FROM forecast_predictions;", conn)
    lsoa_to_msoa = pd.read_sql_query("SELECT * FROM lsoa_msoa_lookup;", conn)
    msoa_force_lookup = pd.read_sql_query(
        "SELECT msoa_code, pfa_code, pfa_name FROM msoa_force_lookup;",
        conn,
    )
    msoa_population = pd.read_sql_query(
        "SELECT msoa_code, population_2024 FROM msoa_population_2024;",
        conn,
    )

forecast = forecast.rename(
    columns={
        "q0.02": "q02",
        "q0.10": "q10",
        "q0.25": "q25",
        "q0.50": "q50",
        "q0.75": "q75",
        "q0.90": "q90",
        "q0.98": "q98",
    }
)

quantile_cols = ["q02", "q10", "q25", "q50", "q75", "q90", "q98"]
quantile_order_ok = forecast[quantile_cols].diff(axis=1).iloc[:, 1:].ge(0).all(axis=1)
print("quantile order fixes:", (~quantile_order_ok).sum())

forecast[quantile_cols] = forecast[quantile_cols].cummax(axis=1)

print("England LSOA info rows:", len(england_lsoa_info))
print("Wales LSOA info rows:", len(wales_lsoa_info))
print("forecast rows:", len(forecast))
forecast.head()


In [ ]:
print("forecast rows:", len(forecast))
print("forecast months:", sorted(forecast["month"].unique()))
print("crime types:")
print(forecast["crime_type"].value_counts().sort_index())

## Add LSOA names, force names, and demographics

In [ ]:
allocation_input_lsoa = forecast.merge(
    lsoa_info[
        [
            "lsoa_code",
            "lsoa_name",
            "loc_auth_code",
            "loc_auth_name",
            "pfa_code",
            "pfa_name",
        ]
    ],
    on="lsoa_code",
    how="left",
)

allocation_input_lsoa = allocation_input_lsoa.merge(
    lsoa_to_msoa[["lsoa_code", "msoa_code", "msoa_name", "lsoa_name_lookup"]],
    on="lsoa_code",
    how="left",
)

allocation_input_lsoa["lsoa_name"] = allocation_input_lsoa["lsoa_name"].fillna(
    allocation_input_lsoa["lsoa_name_lookup"]
)
allocation_input_lsoa["loc_auth_name"] = allocation_input_lsoa["loc_auth_name"].fillna(
    allocation_input_lsoa["msoa_name"].str.replace(r" \d{3}$", "", regex=True)
)
allocation_input_lsoa["loc_auth_code"] = allocation_input_lsoa["loc_auth_code"].fillna("Unknown")

allocation_input_lsoa = allocation_input_lsoa.merge(
    msoa_force_lookup,
    on="msoa_code",
    how="left",
    suffixes=("", "_lookup"),
)
allocation_input_lsoa["pfa_code"] = allocation_input_lsoa["pfa_code"].fillna(
    allocation_input_lsoa["pfa_code_lookup"]
)
allocation_input_lsoa["pfa_name"] = allocation_input_lsoa["pfa_name"].fillna(
    allocation_input_lsoa["pfa_name_lookup"]
)
allocation_input_lsoa = allocation_input_lsoa.drop(
    columns=["pfa_code_lookup", "pfa_name_lookup"]
)

allocation_input_lsoa = allocation_input_lsoa.merge(
    lsoa_demographics[["lsoa_code", "pop"]],
    on="lsoa_code",
    how="left",
)

allocation_input_lsoa.head()

In [ ]:
allocation_input_lsoa[
    [
        "lsoa_name",
        "pfa_code",
        "pfa_name",
        "msoa_code",
        "msoa_name",
        "pop",
    ]
].isna().sum()

## Set crime weights

The crime weights come from `data/cchi_group_average.csv`. This file is the editable input table for crime groups, buckets, CCHI averages, and the final weight used in the model.

In [ ]:
required_cchi_columns = [
    "crime_type",
    "cchi_crime_group",
    "bucket",
    "cchi_match_count",
    "avg_cchi_index",
    "median_cchi_index",
    "min_cchi_index",
    "max_cchi_index",
    "weight",
]

crime_weight_model = pd.read_csv(CCHI_GROUP_AVERAGE_PATH)
missing_columns = [
    col for col in required_cchi_columns
    if col not in crime_weight_model.columns
]
if missing_columns:
    raise ValueError("Missing columns in cchi_group_average.csv: " + ", ".join(missing_columns))

crime_weight_model = crime_weight_model[required_cchi_columns].copy()

if crime_weight_model["crime_type"].duplicated().any():
    duplicated_types = crime_weight_model.loc[
        crime_weight_model["crime_type"].duplicated(),
        "crime_type",
    ].tolist()
    raise ValueError("Duplicate crime types in CCHI input: " + ", ".join(duplicated_types))

crime_bucket_mapping = crime_weight_model.copy()

crime_weight_model

In [ ]:
allocation_input_lsoa = allocation_input_lsoa.merge(
    crime_bucket_mapping,
    on="crime_type",
    how="left",
)

for q in ["q25", "q50", "q75"]:
    allocation_input_lsoa[f"weighted_{q}"] = (
        allocation_input_lsoa[q] * allocation_input_lsoa["weight"]
    )

allocation_input_lsoa["weighted_demand"] = allocation_input_lsoa["weighted_q50"]

allocation_input_lsoa[["bucket", "weight", "weighted_q25", "weighted_q50", "weighted_q75"]].isna().sum()

## Aggregate demand to LSOA and MSOA

In [ ]:
lsoa_bucket_demand = (
    allocation_input_lsoa
    .groupby(
        [
            "lsoa_code",
            "lsoa_name",
            "pfa_code",
            "pfa_name",
            "month",
            "bucket",
        ],
        as_index=False,
    )
    .agg(
        demand_q25=("q25", "sum"),
        demand=("q50", "sum"),
        demand_q75=("q75", "sum"),
        weighted_demand_q25=("weighted_q25", "sum"),
        weighted_demand=("weighted_q50", "sum"),
        weighted_demand_q75=("weighted_q75", "sum"),
        population=("pop", "first"),
    )
)

lsoa_bucket_demand.head()

In [ ]:
print("duplicated LSOA codes:", lsoa_to_msoa["lsoa_code"].duplicated().sum())
print("missing force codes after MSOA lookup:", allocation_input_lsoa["pfa_code"].isna().sum())
lsoa_to_msoa.head()


In [ ]:
rows_before = len(lsoa_bucket_demand)

lsoa_bucket_demand = lsoa_bucket_demand.merge(
    lsoa_to_msoa[["lsoa_code", "msoa_code", "msoa_name"]],
    on="lsoa_code",
    how="left",
)

print("rows before:", rows_before)
print("rows after:", len(lsoa_bucket_demand))
print("missing MSOA codes:", lsoa_bucket_demand["msoa_code"].isna().sum())

In [ ]:
msoa_bucket_demand = (
    lsoa_bucket_demand
    .groupby(
        [
            "pfa_code",
            "pfa_name",
            "msoa_code",
            "month",
            "bucket",
        ],
        as_index=False,
    )
    .agg(
        demand_q25=("demand_q25", "sum"),
        demand=("demand", "sum"),
        demand_q75=("demand_q75", "sum"),
        weighted_demand_q25=("weighted_demand_q25", "sum"),
        weighted_demand=("weighted_demand", "sum"),
        weighted_demand_q75=("weighted_demand_q75", "sum"),
        msoa_name=("msoa_name", "first"),
    )
)

msoa_bucket_demand["msoa_name"] = msoa_bucket_demand["msoa_name"].fillna("Unknown MSOA name")

msoa_bucket_demand.head()

In [ ]:
print("demand total difference:", lsoa_bucket_demand["weighted_demand"].sum() - msoa_bucket_demand["weighted_demand"].sum())
print("duplicated MSOA/month/bucket rows:", msoa_bucket_demand[["pfa_code", "msoa_code", "month", "bucket"]].duplicated().sum())

## Build MSOA context table

In [ ]:
lsoa_context = (
    lsoa_bucket_demand[
        [
            "lsoa_code",
            "lsoa_name",
            "pfa_code",
            "pfa_name",
            "msoa_code",
            "msoa_name",
            "population",
        ]
    ]
    .drop_duplicates(subset=["lsoa_code"])
    .copy()
)

lsoa_context["msoa_name"] = lsoa_context["msoa_name"].fillna("Unknown MSOA name")

msoa_context = (
    lsoa_context
    .groupby(["msoa_code", "msoa_name", "pfa_code", "pfa_name"], as_index=False)
    .agg(
        population=("population", "sum"),
        lsoa_count=("lsoa_code", "count"),
    )
)

msoa_context = msoa_context.merge(msoa_population, on="msoa_code", how="left")
msoa_context["population"] = msoa_context["population"].where(
    msoa_context["population"].notna() & (msoa_context["population"] > 0),
    msoa_context["population_2024"],
)
msoa_context = msoa_context.drop(columns=["population_2024"])

msoa_context.head()


## Add rurality

In [ ]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    lsoa_rurality = pd.read_sql_query(
        "SELECT lsoa_code, ruc_name, urban_rural_flag, is_rural FROM lsoa_rurality;",
        conn,
    )

lsoa_context = lsoa_context.merge(
    lsoa_rurality[["lsoa_code", "is_rural"]],
    on="lsoa_code",
    how="left",
)

msoa_rurality = (
    lsoa_context
    .groupby("msoa_code", as_index=False)
    .agg(
        rurality_score=("is_rural", "mean"),
        rural_lsoa_count=("is_rural", "sum"),
        total_lsoa_count=("lsoa_code", "count"),
    )
)

msoa_context = msoa_context.merge(msoa_rurality, on="msoa_code", how="left")

msoa_context.head()

In [ ]:
print("MSOAs:", len(msoa_context))
print("missing rurality scores:", msoa_context["rurality_score"].isna().sum())
print("forces:")
print(msoa_context["pfa_name"].value_counts())

## Save intermediate tables

The next notebooks read these tables from `allocation_model.db`.

In [ ]:
with sqlite3.connect(ALLOC_DB_PATH) as conn:
    msoa_bucket_demand.to_sql("msoa_bucket_demand", conn, if_exists="replace", index=False)
    msoa_context.to_sql("msoa_context", conn, if_exists="replace", index=False)
    crime_bucket_mapping.to_sql("crime_bucket_mapping", conn, if_exists="replace", index=False)
    crime_weight_model.to_sql("crime_weight_model", conn, if_exists="replace", index=False)

print("Saved:")
print("- msoa_bucket_demand rows:", len(msoa_bucket_demand))
print("- msoa_context rows:", len(msoa_context))
print("- crime_bucket_mapping rows:", len(crime_bucket_mapping))
print("- crime_weight_model rows:", len(crime_weight_model))